# Exploratory data analysis

Label distributions, split sizes, text statistics and graph statistics for every
dataset used in the project. Run `python -m src.prepare_data --stage all` first.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import text_data, prepare_splits
from src.utils import split_labels

DATASETS = ['musiccaps', 'mtat', 'fma_small', 'fma_medium', 'deam']

## 1. Dataset overview

In [ ]:
rows = []
tables = {}
for ds in DATASETS:
    try:
        df, vocab = text_data.load(ds)
        sp = prepare_splits.load(ds)
    except FileNotFoundError:
        print('missing:', ds); continue
    tables[ds] = (df, vocab, sp)
    card = np.mean([len(split_labels(s)) for s in df['labels']])
    rows.append({'dataset': ds, 'clips': len(df), 'tags': len(vocab),
                 'avg tags/clip': round(float(card), 2),
                 'train': len(sp['train']), 'val': len(sp['val']), 'test': len(sp['test']),
                 'avg text chars': int(df['text'].str.len().mean()),
                 'valence/arousal': 'valence' in df.columns})
pd.DataFrame(rows)

## 2. Tag frequency (long-tailed label spaces)

In [ ]:
fig, axes = plt.subplots(len(tables), 1, figsize=(11, 3.1 * len(tables)))
axes = np.atleast_1d(axes)
for ax, (ds, (df, vocab, sp)) in zip(axes, tables.items()):
    counts = df['labels'].str.split('|').explode().value_counts().head(25)
    ax.bar(counts.index, counts.values, color='#4C78A8')
    ax.set_title(f'{ds}: 25 most frequent tags'); ax.tick_params(axis='x', rotation=75, labelsize=7)
plt.tight_layout(); plt.show()

## 3. DEAM valence / arousal distribution

In [ ]:
if 'deam' in tables:
    df = tables['deam'][0]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    h = ax[0].hist2d(df['valence'], df['arousal'], bins=30, cmap='viridis')
    ax[0].set_xlabel('valence'); ax[0].set_ylabel('arousal')
    ax[0].set_title('DEAM affect plane (rescaled to [-1, 1])'); fig.colorbar(h[3], ax=ax[0])
    top = df['labels'].str.split('|').explode().value_counts().head(8).index
    for g in top:
        m = df['labels'].str.contains(g, regex=False)
        ax[1].scatter(df.loc[m, 'valence'], df.loc[m, 'arousal'], s=8, alpha=.55, label=g)
    ax[1].set_xlabel('valence'); ax[1].set_ylabel('arousal')
    ax[1].set_title('affect by genre'); ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.show()

## 4. Graph statistics

In [ ]:
from src.graph_builder import graph_dir, CHORD_NAMES
import torch, collections

rows, chord_hist = [], collections.Counter()
for ds in ['gtzan', 'fma_small', 'mtat', 'deam']:
    shards = sorted(graph_dir(ds).glob('shard_*.pt'))
    if not shards: continue
    recs = torch.load(shards[0], weights_only=True)[:500]
    nodes = [r['num_nodes'] for r in recs]
    edges = [r['edge_index'].shape[1] for r in recs]
    if ds == 'gtzan':
        for r in recs: chord_hist.update(CHORD_NAMES[int(c)] for c in r['chords'])
    rows.append({'dataset': ds, 'graphs sampled': len(recs),
                 'avg nodes': round(np.mean(nodes), 1), 'avg edges': round(np.mean(edges), 1),
                 'avg degree': round(np.mean(edges) / max(np.mean(nodes), 1), 2),
                 'feature dim': int(recs[0]['x'].shape[1])})
pd.DataFrame(rows)

In [ ]:
if chord_hist:
    top = chord_hist.most_common(20)
    plt.figure(figsize=(10, 3.5))
    plt.bar([t[0] for t in top], [t[1] for t in top], color='#54A24B')
    plt.title('GTZAN: most frequent estimated chords'); plt.xticks(rotation=60)
    plt.tight_layout(); plt.show()

## 5. Split integrity (artist leakage)

In [ ]:
pd.DataFrame([{'dataset': ds, **prepare_splits.check_leakage(ds)}
              for ds in tables]).fillna('-')